# 00 — Setup and reproducibility

**Purpose.** Check the public evaluation settings and the checksums of the saved results.
Run this first; notebooks 01–07 inspect the saved artifacts, and notebook 08 provides training steps.

**Inputs** `configs/evaluation_settings.json`, `data_manifest/artifact_checksums.json`,
and the files listed in that checksum manifest.

**Outputs** none; this notebook only reads and verifies files.

Install the packages in `requirements.txt` before running the cells. In Colab, set
`PROVICS_ROOT` to the folder containing this package. Run each notebook from top to bottom.
The saved per-window score arrays contain seed 42; the summary tables include all five seeds.
Notebook 08 prepares raw inputs and fits the recorded ProvICS models in separate Colab cells.

### Reading the saved outputs

The outputs below were produced by executing these cells locally against the released files.
They are saved in this notebook, so GitHub and Colab can show them without another run.
Tables and charts in notebooks 01–07 summarize existing artifacts; no model is trained.
Package versions below describe this replay environment, not the original training environment.
Replaying other notebooks may create disposable copies under `results/regenerated/`.


### Release scope

Notebooks 00–07 check and summarize the saved paper results. Notebook 08 connects the
feature builders and original fitting functions; its training cells are unexecuted in this release.
Alarm-timing and temporal-randomization checks read the saved summaries. Paired score metadata
are under `results/frozen_scores/score_grid/`. The original score archive omits per-view
availability masks; notebook 08 builds masks for newly prepared inputs.

The supplied timestamps place the benign capture after the attack capture. These fixed-reference
results are retrospective and do not establish that the benign reference was available before
the attacks. The separate historical-reference audit is not the primary detector's fitting split.


In [1]:
# Resolve the project root once. Every other path in this notebook is relative to it.
import os, sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    # Point PROVICS_ROOT at your checkout, e.g. a clone inside MyDrive.
    os.environ.setdefault('PROVICS_ROOT', '/content/drive/MyDrive/provics-recording-aware-evaluation')
else:
    os.environ.setdefault('PROVICS_ROOT', str(Path.cwd().parent))

sys.path.insert(0, os.environ['PROVICS_ROOT'])
from src.utils.paths import project_root
ROOT = project_root()
print('Project folder:', ROOT.name)

# Render compact, readable tables below each cell.
from IPython.display import display
import pandas as pd
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 12)
pd.set_option('display.precision', 4)


Project folder: code


In [2]:
# Environment. Seeds are recorded rather than set globally: each fitted model in the study
# carried its own seed, and those are listed in the evaluation settings.
import platform, numpy, pandas, sklearn
print('python      ', platform.python_version())
for module in (numpy, pandas, sklearn):
    print(f'{module.__name__:<12}', module.__version__)


python       3.12.2
numpy        1.26.4
pandas       2.2.1
sklearn      1.9.0


In [3]:
# Settings needed to interpret the released artifacts; hashes are stored separately.
from src.evaluation.replay import load_settings
settings = load_settings()
print('schema       :', settings['schema_version'])
print('paper        :', settings['paper']['title'])
print('status       :', settings['paper']['status'])
print('dataset      :', settings['dataset']['huggingface_repo'])
print('pinned rev   :', settings['dataset']['pinned_revision'])
print('seeds        :', settings['detector']['seeds'])
print('score seed   :', settings['detector']['frozen_score_seed'])
print('windows (s)  :', settings['windowing']['durations_seconds'])
print('buffer (s)   :', settings['windowing']['same_recording_background_buffer_seconds'])
display(pd.DataFrame({'Setting': ['Window sizes (s)', 'Fit seeds', 'Saved score seed', 'Background buffer (s)'], 'Value': [str(settings['windowing']['durations_seconds']), str(settings['detector']['seeds']), str(settings['detector']['frozen_score_seed']), str(settings['windowing']['same_recording_background_buffer_seconds'])]}))


schema       : 1.1
paper        : Recording-Aware Evaluation of Multimodal OT Intrusion Detection
status       : Accepted for presentation at ICSS 2026
dataset      : trucyberlab/multimodal-ICS-provenance
pinned rev   : 18b4b0e1359d3d02347301c55f97c3d98f5dab5c
seeds        : [42, 43, 44, 45, 46]
score seed   : 42
windows (s)  : [5, 10, 30, 60]
buffer (s)   : 300


,Setting,Value
0,Window sizes (s),"[5, 10, 30, 60]"
1,Fit seeds,"[42, 43, 44, 45, 46]"
2,Saved score seed,42
3,Background buffer (s),300


In [4]:
# Verify each released artifact against its recorded checksum.
from collections import Counter
from src.evaluation.replay import verify_artifacts
records = verify_artifacts()
print(Counter(r['status'] for r in records))
for record in records:
    if record['status'] != 'ok':
        print(' ', record['status'], record['artifact'])
assert all(r['status'] == 'ok' for r in records), 'released artifacts are not intact'
print(f'\nall {len(records)} artifacts intact')
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame(records)[['artifact', 'status']])
print('The table lists all files; a long display may scroll horizontally.')


Counter({'ok': 67})

all 67 artifacts intact


,artifact,status
0,configs/evaluation_settings.json,ok
1,configs/frozen_specs/config_revision3.json,ok
2,configs/frozen_specs/config_revision3.sha256,ok
3,configs/frozen_specs/feature_lists_post_screen...,ok
4,configs/frozen_specs/m4_variables.json,ok
5,configs/frozen_specs/protocol_v5.json,ok
6,configs/frozen_specs/protocol_v5.sha256,ok
7,configs/frozen_specs/protocol_v5_addendum_A1.json,ok
8,configs/frozen_specs/protocol_v5_addendum_A1.s...,ok
9,configs/frozen_specs/supervisor_buffer_sensiti...,ok


The table lists all files; a long display may scroll horizontally.


In [5]:
# What is canonical, what is optional, and what is not shipped.
for label, path in [('canonical tables ', 'results/paper_tables'),
                    ('canonical figures', 'results/paper_figures'),
                    ('frozen replay in ', 'results/frozen_scores'),
                    ('frozen specs     ', 'configs/frozen_specs')]:
    files = sorted(p for p in (ROOT / path).rglob('*') if p.is_file())
    print(f'{label} {len(files):>3} files  {path}')
print('\nregenerated output goes to results/regenerated/ and is never mixed with the above.')
display(pd.DataFrame([{'Folder': name, 'Files': sum(p.is_file() for p in (ROOT / name).rglob('*'))} for name in ['results/paper_tables', 'results/paper_figures', 'results/frozen_scores', 'configs/frozen_specs']]))


canonical tables   23 files  results/paper_tables
canonical figures   3 files  results/paper_figures
frozen replay in   27 files  results/frozen_scores
frozen specs       10 files  configs/frozen_specs

regenerated output goes to results/regenerated/ and is never mixed with the above.


,Folder,Files
0,results/paper_tables,23
1,results/paper_figures,3
2,results/frozen_scores,27
3,configs/frozen_specs,10


## What this notebook checked

- The public settings match their recorded checksum.
- Every file in the artifact manifest is present and unchanged.
- Notebook outputs belong in `results/regenerated/`, separate from the saved results.

A matching checksum establishes file integrity, not the correctness of an analysis.
